In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [22]:
df = pd.read_csv("PDCS8102_Topic4_Service_Centre_Raw.csv")
df.head()

,record_id,district,service_type,waiting_minutes,satisfaction,visit_date
0,S001,Blantyre,Licensing,15.0,2.0,2026-09-02
1,S002,Mzuzu,Records,25.0,3.0,2026-09-03
2,S003,lilongwe,Registration,34.0,4.0,2026-09-04
3,S004,MZ,Licensing,45.0,NaN,2026-09-05
4,S005,BLANTYRE,Records,NaN,1.0,2026-09-06


In [23]:
print("Rows and Columns:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData types:") 
print(df.dtypes) 
print("\nFirst five records:") 
print(df.head()) 

Rows and Columns: (32, 6)

Column Names: ['record_id', 'district', 'service_type', 'waiting_minutes', 'satisfaction', 'visit_date']

Data types:
record_id           object
district            object
service_type        object
waiting_minutes    float64
satisfaction       float64
visit_date          object
dtype: object

First five records:
  record_id   district  service_type  waiting_minutes  satisfaction  \
0      S001   Blantyre     Licensing             15.0           2.0   
1      S002      Mzuzu       Records             25.0           3.0   
2      S003  lilongwe   Registration             34.0           4.0   
3      S004         MZ     Licensing             45.0           NaN   
4      S005   BLANTYRE       Records              NaN           1.0   

   visit_date  
0  2026-09-02  
1  2026-09-03  
2  2026-09-04  
3  2026-09-05  
4  2026-09-06  


# **Check missing values and repeated rows**

In [32]:
print("Missing values in each column:") 
print(df.isnull().sum())
print("\nMissing percentages:") 
print((df.isnull().mean() * 100).round(2))
print("\nExact duplicate rows:", df.duplicated().sum()) 
print("\nDifferent district labels:") 
print(df["district"].unique()) 

Missing values in each column:
record_id          0
district           0
service_type       0
waiting_minutes    3
satisfaction       3
visit_date         0
dtype: int64

Missing percentages:
record_id          0.00
district           0.00
service_type       0.00
waiting_minutes    9.38
satisfaction       9.38
visit_date         0.00
dtype: float64

Exact duplicate rows: 2

Different district labels:
['Blantyre' 'Mzuzu' 'lilongwe ' 'MZ' 'BLANTYRE' 'Lilongwe']


Checking for outliers

In [35]:
print(df["waiting_minutes"].describe()) 
print(df.loc[df["waiting_minutes"] < 0])
print(df.loc[df["waiting_minutes"] > 120])

count     29.000000
mean      45.931034
std       73.242713
min      -12.000000
25%       15.000000
50%       22.000000
75%       45.000000
max      300.000000
Name: waiting_minutes, dtype: float64
   record_id   district  service_type  waiting_minutes  satisfaction  \
26      S027  lilongwe   Registration            -12.0           3.0   

    visit_date  
26  2026-09-28  
   record_id district service_type  waiting_minutes  satisfaction  visit_date
15      S016       MZ    Licensing            300.0           2.0  2026-09-17
31      S016       MZ    Licensing            300.0           2.0  2026-09-17


**Activity 1 observations**

*Make a working copy so the original stays unchanged*

In [36]:
clean = df.copy()
print ("Original rows:", len(df))
print ("Working-copy rows:", len(clean))

Original rows: 32
Working-copy rows: 32


*Remove only exact duplicate rows*

In [37]:
clean = clean.drop_duplicates()
print ("Rows after dropping duplicates:", len(clean))

Rows after dropping duplicates: 30


*Treat impossible negative waiting times as missing*

In [38]:
clean.loc[clean["waiting_minutes"] < 0, "waiting_minutes"] = np.nan
print(clean["waiting_minutes"].isna().sum())

4


*Compare the mean and median before deciding*

In [40]:
print("Mean:", clean["waiting_minutes"].mean().round(2))
print("Median:", clean["waiting_minutes"].median())

Mean: 39.69
Median: 23.5


In [43]:
#Create a missing-value flag and fill missing waiting times
clean["waiting_was_missing"] = clean["waiting_minutes"].isna() 
median_wait = clean["waiting_minutes"].median() 
clean["waiting_minutes_filled"] = clean["waiting_minutes"].fillna(median_wait) 
print(clean[["waiting_minutes", "waiting_was_missing", 
    "waiting_minutes_filled"]].head(10)) 

   waiting_minutes  waiting_was_missing  waiting_minutes_filled
0             15.0                False                    15.0
1             25.0                False                    25.0
2             34.0                False                    34.0
3             45.0                False                    45.0
4              NaN                 True                    23.5
5             75.0                False                    75.0
6             12.0                False                    12.0
7             18.0                False                    18.0
8             22.0                False                    22.0
9              8.0                False                     8.0


In [42]:
#Keep missing satisfaction scores missing 
print("Missing satisfaction scores:", clean["satisfaction"].isna().sum()) 

Missing satisfaction scores: 3


### Activity 2: Missing values and duplicates

**(a) Why a working copy was used**

A working copy was created using `clean = df.copy()` to preserve the original dataset. This allows changes to be checked against the original records and mistakes to be corrected without altering the source data.

**(b) Why an exact duplicate might be removed**

An exact duplicate contains identical values in every column and may represent the same service encounter recorded twice. Removing it prevents double counting, which could distort totals, averages and other results. However, a repeated identifier alone does not prove duplication, because the same person may have made separate visits.

**(c) Why missing satisfaction should not automatically be replaced by a neutral score**

A missing satisfaction score indicates that the customer’s opinion is unknown, while a neutral score represents an expressed opinion. Replacing missing values with neutral scores would introduce responses that customers did not provide and could bias the results. Therefore, missing satisfaction scores were retained as missing.

In [45]:
#Calculate the IQR boundaries 
q1 = clean["waiting_minutes_filled"].quantile(0.25) 
q3 = clean["waiting_minutes_filled"].quantile(0.75) 
iqr = q3 - q1 
lower = q1 - 1.5 * iqr 
upper = q3 + 1.5 * iqr 
print("Lower bound:", lower, "Upper bound:", upper) 

Lower bound: -11.625 Upper bound: 61.375


In [46]:
#Flag unusual waiting times without deleting them 
clean["possible_outlier"] = ( 
(clean["waiting_minutes_filled"] < lower) | 
(clean["waiting_minutes_filled"] > upper) 
) 
print(clean.loc[clean["possible_outlier"], 
    ["record_id", "waiting_minutes_filled"]]) 

   record_id  waiting_minutes_filled
5       S006                    75.0
15      S016                   300.0
25      S026                    75.0


In [47]:
#Standardize district names 
print("Before:", clean["district"].unique()) 
clean["district"] = clean["district"].str.strip().str.title() 
clean["district"] = clean["district"].replace({"Mz": "Mzuzu"}) 
print("After:", clean["district"].unique()) 

Before: ['Blantyre' 'Mzuzu' 'lilongwe ' 'MZ' 'BLANTYRE' 'Lilongwe']
After: ['Blantyre' 'Mzuzu' 'Lilongwe']


**Why the 300-minute waiting time should be investigated**

The 300-minute waiting time is unusually high, but it is not automatically an error. It may represent a genuine delay caused by service disruptions or high demand. The record should be verified before deletion, because removing a valid observation could underestimate waiting times and hide service-delivery problems. It should therefore be flagged as a possible outlier until investigated.

**Standardised district labels**

After removing extra spaces, standardising capitalisation and replacing Mz with Mzuzu, the district labels are:
- Blantyre
- Mzuzu
- Lilongwe

**E. Lab activity 4 - Feature engineering, encoding and scaling (30 minutes)**

In [48]:
#Convert dates from text into dates 
clean["visit_date"] = pd.to_datetime(clean["visit_date"], errors="coerce") 
print(clean[["visit_date"]].head())

  visit_date
0 2026-09-02
1 2026-09-03
2 2026-09-04
3 2026-09-05
4 2026-09-06


In [49]:
#Create a weekday feature 
clean["weekday"] = clean["visit_date"].dt.day_name() 
print(clean[["visit_date", "weekday"]].head()) 

  visit_date    weekday
0 2026-09-02  Wednesday
1 2026-09-03   Thursday
2 2026-09-04     Friday
3 2026-09-05   Saturday
4 2026-09-06     Sunday


In [51]:
#Create waiting-time categories 
clean["wait_category"] = pd.cut( 
clean["waiting_minutes_filled"], 
bins=[-0.01, 20, 60, float("inf")], 
labels=["Short", "Moderate", "Long"] 
) 
print(clean[["waiting_minutes_filled", "wait_category"]].head()) 

   waiting_minutes_filled wait_category
0                    15.0         Short
1                    25.0      Moderate
2                    34.0      Moderate
3                    45.0      Moderate
4                    23.5      Moderate


In [52]:
#17. One-hot encode district 
district_columns = pd.get_dummies(clean["district"], prefix="district", dtype=int) 
print(district_columns.head()) 

   district_Blantyre  district_Lilongwe  district_Mzuzu
0                  1                  0               0
1                  0                  0               1
2                  0                  1               0
3                  0                  0               1
4                  1                  0               0


In [53]:
#18. Standardize filled waiting times 
scaler = StandardScaler() 
clean["waiting_scaled_demo"] = scaler.fit_transform( 
clean[["waiting_minutes_filled"]] 
).ravel() 
print(clean[["waiting_minutes_filled", "waiting_scaled_demo"]].head())

   waiting_minutes_filled  waiting_scaled_demo
0                    15.0            -0.433535
1                    25.0            -0.241138
2                    34.0            -0.067980
3                    45.0             0.143657
4                    23.5            -0.269998


### Activity 4: Feature engineering, encoding and scaling

**Weekday:** The `weekday` column identifies the day of the week on which each visit occurred, based on `visit_date`. It allows waiting times and service demand to be compared across weekdays.

**Waiting-time category:** The `wait_category` column groups filled waiting times into three categories: **Short** (0–20 minutes), **Moderate** (more than 20 up to 60 minutes), and **Long** (more than 60 minutes). These are illustrative categories rather than official service-quality standards.

**One-hot columns:** The `district_Blantyre`, `district_Lilongwe` and `district_Mzuzu` columns represent district membership. A value of `1` indicates that a record belongs to that district, while `0` indicates that it does not. This represents districts numerically without implying an order or ranking.

**Negative scaled values:** A value in `waiting_scaled_demo` is negative when the filled waiting time is below the mean waiting time used by the scaler. It does not indicate a negative actual waiting time. For example, a scaled value of `−1` means the waiting time is one standard deviation below the mean.

***F. Lab activity 5 - Integration, validation and export (20 minutes)**

In [55]:
#19. Load the district lookup file 
lookup = pd.read_csv("PDCS8102_Topic4_District_Lookup.csv") 
print(lookup) 

   district    region
0  Lilongwe   Central
1  Blantyre  Southern
2     Mzuzu  Northern


In [58]:
#20. Combine the two tables using the district column 
merged = clean.merge( 
lookup, 
on="district", 
how="left", 
validate="many_to_one", 
indicator=True 
) 
print(merged[["record_id", "district", "region", "_merge"]].head())

  record_id  district    region _merge
0      S001  Blantyre  Southern   both
1      S002     Mzuzu  Northern   both
2      S003  Lilongwe   Central   both
3      S004     Mzuzu  Northern   both
4      S005  Blantyre  Southern   both


In [59]:
#21. Check the result 
print("Rows before merge:", len(clean)) 
print("Rows after merge:", len(merged)) 
print("Unmatched districts:") 
print(merged.loc[merged["_merge"] != "both", "district"].unique()) 

Rows before merge: 30
Rows after merge: 30
Unmatched districts:
[]


In [61]:
#22. Export the prepared dataset 
prepared = merged.drop(columns=["_merge"]) 
prepared.to_csv("Topic4_Service_Centre_My_Prepared_Data.csv", index=False) 
print("Saved prepared CSV successfully.") 


Saved prepared CSV successfully.


### Tutorial responses

1. **Could deleting all rows with missing data create bias?**  
   Yes. If missing values occur more frequently among particular districts, service types or customer groups, deleting those rows could underrepresent those groups and bias the results. It also reduces the available sample size.

2. **When might a repeated identifier be legitimate?**  
   A repeated identifier may be legitimate when the same customer makes multiple visits or receives different services. A repeated identifier alone does not prove that records are duplicates.

3. **How could an incorrect district mapping affect results?**  
   An incorrect mapping could assign records to the wrong district or region, producing misleading totals, averages and comparisons. It could also prevent records from matching the district lookup table.

4. **What is the difference between a flagged outlier and a confirmed error?**  
   A flagged outlier is an unusual value that requires investigation but may be valid, such as a genuine 300-minute wait. A confirmed error is a value verified as incorrect through source records or other reliable evidence.

5. **Why should scaling parameters be learned only from training data?**  
   Learning the mean and standard deviation from training data only prevents data leakage, where information from the test set influences model preparation. The fitted scaler is then applied to the test data, preserving an independent evaluation of model performance.